# 🎬 YouTube Átirat Letöltő Pro (Google Colab Felhő)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krisatrader/YT-text-downloader/blob/main/youtube_transcript_downloader_colab.ipynb)

Tölts le teljes YouTube lejátszási listákat, csatornákat vagy videókat **.md** és **.txt** formátumban automatikus fordítással és emberi sebességű robotvédelemmel (min. 3.0 mp késleltetés)!

---

In [ ]:
#@title 🚀 1. Telepítés és Előkészítés (Kattints ide a futtatáshoz ▶️)
#@markdown Ez a lépés letölti a kódot a GitHub-ról és feltelepíti a szükséges csomagokat.

import os, sys

!rm -rf /content/YT-text-downloader
!git clone https://github.com/krisatrader/YT-text-downloader.git /content/YT-text-downloader
%cd /content/YT-text-downloader
!pip install -q -r requirements.txt

print("\n✅ Sikeres előkészítés! Töltsd ki az alábbi 2/A űrlapot és kattints a futtatásra.")

In [ ]:
#@title ⚡ 2/A. 1-Kattintásos Letöltés Űrlap (Minden beállítás egy helyen)
#@markdown **Töltsd ki az űrlapot, majd kattints a bal oldali ▶️ gombra.** A letöltött ZIP automatikusan megérkezik a gépedre!

YouTube_URL = "https://youtube.com/playlist?list=PLKfy8g4yBtRY&si=IO9L0gnJPgCAurG_" #@param {type:"string"}
Formatum = "both" #@param ["both", "md", "txt"]
Preferalt_Nyelvek = "hu,en" #@param {type:"string"}
Forditas_Celnyelve = "original" #@param ["original", "hu", "en", "de", "es", "fr", "it", "egyéni"]
Egyeni_Celnyelv_Kod = "" #@param {type:"string"}
Idobelyegek = True #@param {type:"boolean"}
#@markdown **Késleltetés másodpercben (Alapértelmezett: 3.0-5.0 mp, a biztonság kedvéért 3.0 mp alá nem csökkenthető):**
Emberi_Keses_Masodperc = "3.0-5.0" #@param {type:"string"}
Max_Videok_Limit = 0 #@param {type:"integer"}

#@markdown **Opcionális Védelem (Proxy & Sütik):**
Proxy_Szerver = "" #@param {type:"string"}
Sutik_Netscape_Szoveg = "" #@param {type:"string"}

import os, sys, zipfile
from google.colab import files

%cd /content/YT-text-downloader
from transcript_downloader import DownloaderEngine, TranscriptTranslator, CookieManager, sanitize_filename

# 1. Sütik beállítása (ha megadva)
if Sutik_Netscape_Szoveg.strip():
    cm = CookieManager("transcripts_output/.saved_cookies.txt")
    cm.save_cookie_text(Sutik_Netscape_Szoveg.strip())
    print("🍪 Sütik sikeresen betöltve!")

# 2. Célnyelv eldöntése
target_lang = Forditas_Celnyelve
if Forditas_Celnyelve == "egyéni" and Egyeni_Celnyelv_Kod.strip():
    target_lang = Egyeni_Celnyelv_Kod.strip()

# 3. Késleltetés feldolgozása (Biztonsági alsó határ betartatása: minimum 3.0 másodperc)
delay_min, delay_max = 3.0, 5.0
if "-" in Emberi_Keses_Masodperc:
    try:
        dparts = Emberi_Keses_Masodperc.split("-")
        delay_min = max(3.0, float(dparts[0]))
        delay_max = max(delay_min, float(dparts[1]))
    except Exception:
        pass
elif Emberi_Keses_Masodperc.strip():
    try:
        val = max(3.0, float(Emberi_Keses_Masodperc.strip()))
        delay_min, delay_max = val, val + 2.0
    except Exception:
        pass

proxy_val = Proxy_Szerver.strip() if Proxy_Szerver.strip() else None

engine = DownloaderEngine(
    output_dir="transcripts_output",
    output_format=Formatum,
    target_language=target_lang,
    delay_range=(delay_min, delay_max),
    preferred_languages=[l.strip() for l in Preferalt_Nyelvek.split(",") if l.strip()],
    include_timestamps=Idobelyegek,
    limit=Max_Videok_Limit if Max_Videok_Limit > 0 else None,
    proxy=proxy_val,
)

print(f"🚀 Letöltés indítása emberi tempóval ({delay_min:.1f}-{delay_max:.1f} mp + 20 videónként pihenő): {YouTube_URL}...")
res = engine.run(YouTube_URL)

if res and res.get("collection_title"):
    col_title = res["collection_title"]
    clean_title = sanitize_filename(col_title)
    target_dir = os.path.join("transcripts_output", clean_title)
    zip_filename = f"{clean_title}.zip"

    with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, filenames in os.walk(target_dir):
            for fname in filenames:
                fpath = os.path.join(root, fname)
                zf.write(fpath, os.path.relpath(fpath, target_dir))

    print(f"\n📦 ZIP fájl elkészült: {zip_filename}")
    print("⬇️ Letöltés kezdeményezése a böngésződben...")
    files.download(zip_filename)
else:
    print("⚠️ Nem sikerült a gyűjtemény letöltése vagy nincsenek videók.")

In [ ]:
#@title 🌐 2/B. Webes Kezelőfelület Indítása (Opcionális Sötét Módú Web UI)
#@markdown Ha a teljes vizuális webes kezelőfelületet szeretnéd futtatni a felhőben.

import os, sys, time, subprocess, re, urllib.request
from google.colab.output import serve_kernel_port_as_window

%cd /content/YT-text-downloader

# 1. Web szerver indítása a háttérben
print("🚀 Web szerver indítása...")
web_proc = subprocess.Popen([sys.executable, "web_app.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

print("\n" + "="*65)
print("🎉 1. OPCIÓ (Google Colab Közvetlen Link):")
try:
    serve_kernel_port_as_window(8000, anchor_text="👉 KATTINTS IDE A WEB FELÜLET MEGNYITÁSÁHOZ (Google Colab)")
except Exception as e:
    print(f"Colab port: {e}")

# 2. Cloudflared letöltése tartalék publikus linkhez
cf_path = "/usr/local/bin/cloudflared"
if not os.path.exists(cf_path):
    try:
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf_path)
        os.chmod(cf_path, 0o777)
    except Exception:
        pass

if os.path.exists(cf_path):
    tunnel_proc = subprocess.Popen([cf_path, "tunnel", "--url", "http://127.0.0.1:8000"], stderr=subprocess.PIPE, text=True)
    for line in tunnel_proc.stderr:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            print("\n🎉 2. OPCIÓ (Cloudflare Publikus Link):")
            print(f"👉 {match.group(0)}")
            break
print("="*65 + "\n")

# Szerver életben tartása
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Szerver leállítva.")